# Ring-attractor RNN — a full geometry & topology walkthrough

This notebook works **one example network in detail**: a continuous-time rate RNN
with **ring-attractor** connectivity, driven by a stimulus bump that rotates once
around the ring during the trial. We compute the entire `neuralgeom` *subspace
lens* on it and narrate what each quantity describes.

**Why a ring attractor?** Its recurrent connectivity `W_ij = (J₁/N)·cos(θᵢ−θⱼ)`
supports a *continuum* of stable bump states arranged on a circle. When the input
bump rotates once, the population activity bump follows it once around the ring,
so the **coding direction** (the axis the activity lies along) rotates through a
full turn. That is a loop — and the natural place to *see* it is not the raw
state but the **subspace** the activity occupies.

**The one move.** Instead of the raw state `x(t) ∈ ℝᴺ`, track the `k`-dimensional
subspace the activity locally occupies as a single point on the Grassmannian
manifold `Gr(k, N)`. For `k = 1`, `Gr(1, N) = ℝPᴺ⁻¹` (real projective space).
As the coding axis rotates, that point traces a closed curve on `Gr(1, N)`.

**What we expect to be able to describe** (and will test):
- a **persistent loop** in the topology at `k = 1` (an H1 feature) but **not** at
  `k = 2` — `k` acts as a *topological filter*;
- **uniform-circular-motion** kinematics — near-constant subspace speed and
  curvature, nonzero covariant acceleration;
- **intrinsic dimensionality 2** from tangent-PCA;
- high **frame reliability** (`σ_k/σ_{k+1} ≫ 1`) at `k = 1`.

Terms are defined the first time they appear.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spawn" if False else "axes.axisbelow": True})
RNG = np.random.default_rng(0)

import neuralgeom as ng
print("neuralgeom", ng.__version__)

## 1. Generate the example network

`SubspaceRNNConfig(connectivity="ring", ring_moving=True)` builds the ring RNN and
drives it with a cosine bump making **one full revolution** over the trial. The
result is a `Trajectory` — the one object every analysis consumes
(`X` = states `(n_trials, T, N)`, `time`, plus connectivity `W`, the per-trial
inputs, and the ring angles `θ` in `aux`).

In [ ]:
from neuralgeom.synth.subspace_rnn import SubspaceRNNConfig, make_trajectory

cfg = SubspaceRNNConfig(connectivity="ring", ring_moving=True, ring_revolutions=1.0,
                        N=60, n_trials=12, duration=1.5, dt=2e-3,
                        ring_J1=4.0, ring_stim_amp=1.0, noise_std=0.05, seed=1)
traj = make_trajectory(cfg)
theta = traj.aux["theta"]              # preferred angle of each unit on the ring
print(traj)
print("units carry ring angles θ:", theta.shape, "spanning [0, 2π)")

## 2. Start from the raw data

Following good exploratory practice, look at the rawest view first: individual
unit traces and the population as a heatmap, before any dimensionality reduction.

In [ ]:
tr = 0
X = traj.X[tr]                          # (T, N) one trial
t = traj.time
order = np.argsort(theta)               # sort units by preferred angle

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
# (a) a handful of single-unit rate traces
for u in np.linspace(0, traj.N-1, 6).astype(int):
    ax[0].plot(t, np.tanh(X[:, u]), lw=1.2, label=f"unit {u} (θ={theta[u]:.1f})")
ax[0].set(title="Single-unit activity r=tanh(x)  (trial 0)", xlabel="time (s)",
          ylabel="rate"); ax[0].legend(fontsize=7, ncol=2)
# (b) population heatmap, units sorted by preferred angle → a travelling bump
im = ax[1].imshow(np.tanh(X[:, order]).T, aspect="auto", origin="lower",
                  extent=[t[0], t[-1], 0, traj.N], cmap="magma")
ax[1].set(title="Population bump (units sorted by θ) — it travels once around",
          xlabel="time (s)", ylabel="unit (sorted by θ)")
fig.colorbar(im, ax=ax[1], label="rate"); fig.tight_layout(); display(fig); plt.close(fig)

The heatmap shows a **travelling bump**: the active population marches through
unit-index (i.e. through preferred angle θ) once across the trial. That is the
ring turning once. Now we ask where that shows up geometrically.

## 3. Raw state-space geometry (PCA)

**PCA** (principal component analysis) finds the orthogonal directions of greatest
variance in the `N`-dimensional state. Projecting the trajectory onto the top 2–3
PCs is the standard first look at population geometry.

In [ ]:
from numpy.linalg import svd
Xc = X - X.mean(0)
U, S, Vt = svd(Xc, full_matrices=False)
pcs = Xc @ Vt[:3].T

fig = plt.figure(figsize=(13, 4.4))
ax0 = fig.add_subplot(1, 3, 1, projection="3d")
sctt = ax0.scatter(pcs[:,0], pcs[:,1], pcs[:,2], c=t, cmap="viridis", s=8)
ax0.plot(pcs[:,0], pcs[:,1], pcs[:,2], color="0.8", lw=0.5)
ax0.set(title="Trial 0 state trajectory (PCA 3)", xlabel="PC1", ylabel="PC2"); ax0.set_zlabel("PC3")
# several trials overlaid in PC1-2
ax1 = fig.add_subplot(1, 3, 2)
for i in range(min(6, traj.n_trials)):
    Xi = traj.X[i] - traj.X[i].mean(0)
    p = Xi @ Vt[:2].T
    ax1.plot(p[:,0], p[:,1], lw=1, alpha=0.8)
ax1.set(title="6 trials in PC1–PC2 (different start phases)", xlabel="PC1", ylabel="PC2")
# participation ratio per trial
def participation_ratio(A):
    A = A - A.mean(0); C = A.T @ A / A.shape[0]
    ev = np.clip(np.linalg.eigvalsh(C), 0, None)
    return ev.sum()**2 / (np.sum(ev**2) + 1e-12)
pr = [participation_ratio(traj.X[i]) for i in range(traj.n_trials)]
ax2 = fig.add_subplot(1, 3, 3)
ax2.bar(range(traj.n_trials), pr, color="C0"); ax2.axhline(np.mean(pr), color="C3", ls="--")
ax2.set(title=f"Participation ratio (mean {np.mean(pr):.1f})", xlabel="trial", ylabel="PR")
fig.tight_layout(); display(fig); plt.close(fig)

The single-trial trajectory is a **ring in state space**, and across trials the
rings are rotated copies (different bump start phases). Participation ratio — an
estimate of the *effective number of dimensions* (≈ how many PCs the variance
spreads across) — sits near 2–3, consistent with a 2-D plane carrying a circle.
That is the state-space view. The **subspace** view asks a sharper question:
does the *coding direction itself* rotate?

## 4. Grassmannian embedding & frame reliability

We slide a short window along the trial; the top-`k` right singular vectors of the
windowed states form an orthonormal **frame** `(N, k)` — the `k`-dim subspace the
activity occupies there, i.e. a point on `Gr(k, N)`.

Two diagnostics, defined here:
- **`evr`** (explained-variance ratio): fraction of window variance captured by
  the top-`k` directions — how well a `k`-frame summarises the window.
- **`sv_gap` = σ_k/σ_{k+1}**: the **frame-reliability** ratio. A `k`-frame is only
  trustworthy when this is comfortably above 1; near 1 means the `k`-th direction
  is indistinguishable from noise.

In [ ]:
from neuralgeom.subspace import EmbedConfig, embed_from_trajectory

emb1 = embed_from_trajectory(traj, tr, EmbedConfig(k=1, win=60, stride=10))
emb2 = embed_from_trajectory(traj, tr, EmbedConfig(k=2, win=60, stride=10))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(emb1["win_times"], emb1["evr"], label="k=1", color="C0")
ax[0].plot(emb2["win_times"], emb2["evr"], label="k=2", color="C1")
ax[0].set(title="Top-k variance fraction (evr)", xlabel="time (s)", ylabel="evr"); ax[0].legend()
ax[1].semilogy(emb1["win_times"], emb1["sv_gap"], label="k=1  σ₁/σ₂", color="C0")
ax[1].semilogy(emb2["win_times"], emb2["sv_gap"], label="k=2  σ₂/σ₃", color="C1")
ax[1].axhline(1.0, color="0.4", ls=":")
ax[1].set(title="Frame reliability  σ_k/σ_{k+1}", xlabel="time (s)", ylabel="ratio (log)"); ax[1].legend()
fig.tight_layout(); display(fig); plt.close(fig)
print(f"median sv_gap: k=1 → {np.median(emb1['sv_gap']):.1f}   k=2 → {np.median(emb2['sv_gap']):.1f}")

**Reading it.** Both frames are *reliable* here — the instantaneous window is essentially rank ≤ 2 (σ₃ and beyond are tiny), so `k=1` has a large σ₁/σ₂ gap (one dominant coding direction at each instant) and `k=2` has an even larger σ₂/σ₃ gap. So the `k=1`-vs-`k=2` difference we find later is **not** a reliability artifact — it is genuinely *topological*: whether the tracked subspace *moves*. (Contrast with a point attractor, where the activity really is rank-1 and `k=2` would track noise — that is the case `sv_gap` is designed to catch.)

## 5. Subspace drift and recurrence

**Geodesic distance** on `Gr(k,N)` measures how different two subspaces are
(built from the principal angles between them; the canonical metric = √2 ×
arc-length). Two views:
- **drift** `d(frame₀, frameₜ)` — how far the subspace has moved from its start;
- the **self-distance matrix** `D[i,j] = d(frameᵢ, frameⱼ)` — a *recurrence plot*:
  dark off-diagonal bands mean the subspace **returned** to an earlier orientation
  (the signature of a loop).

In [ ]:
from neuralgeom.subspace.embed import subspace_drift
from neuralgeom.geometry import frame_distance_matrix

D1 = frame_distance_matrix(emb1["frames"], "canonical")
D2 = frame_distance_matrix(emb2["frames"], "canonical")
tc = emb1["win_times"]

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].plot(tc, subspace_drift(emb1["frames"]), color="C0", label="k=1")
ax[0].plot(emb2["win_times"], subspace_drift(emb2["frames"]), color="C1", label="k=2")
ax[0].set(title="Subspace drift d(frame₀, frameₜ)", xlabel="time (s)", ylabel="geodesic dist"); ax[0].legend()
for a, D, lab in ((ax[1], D1, "k=1"), (ax[2], D2, "k=2")):
    im = a.imshow(D, origin="lower", cmap="magma", extent=[tc[0], tc[-1], tc[0], tc[-1]])
    a.set(title=f"Self-distance (recurrence), {lab}", xlabel="time (s)", ylabel="time (s)")
    fig.colorbar(im, ax=a, fraction=0.046)
fig.tight_layout(); display(fig); plt.close(fig)

At **k = 1** the drift rises and then **returns toward zero**, and the recurrence
matrix shows the tell-tale off-diagonal valley of a closed orbit: the coding line
comes back to (near) where it started — a loop on `ℝPᴺ⁻¹`. At **k = 2** the recurrence shows little structure and small distances: the containing 2-plane is roughly *static* while only the *direction within it* rotates, so the 2-frame barely moves and no loop forms — even though the 2-frame is perfectly reliable.

## 6. Riemannian kinematics — how the subspace moves

Treating the frames as a curve on the manifold, we compute (all intrinsic, using
the manifold's own log/exp/parallel-transport):
- **speed** `‖vₜ‖` = geodesic distance covered per second;
- **covariant acceleration** `‖aₜ‖` — change in velocity *after* correcting for the
  curved manifold (parallel transport). It is **zero on a geodesic**, so nonzero
  `‖a‖` means the path bends;
- **curvature** `κ = ‖a_⊥‖ / speed²` — the geometric bending rate.

A perfect uniform rotation should give near-**constant speed**, near-**constant
curvature**, and **nonzero constant** covariant acceleration (a circle is not a
geodesic). The identity `speed · dt == step_dist` is a built-in correctness
check.

In [ ]:
from neuralgeom.subspace import KinConfig, compute_kinematics
kin = compute_kinematics(emb1["frames"], emb1["win_times"], KinConfig())
ident = np.nanmax(np.abs(kin["speed"]*kin["dt"] - kin["step_dist"]))

fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
ax[0].plot(tc, kin["speed"], color="C0"); ax[0].set(title="Subspace speed ‖v‖", xlabel="time (s)")
ax[1].plot(tc, kin["acc_mag"], color="C3"); ax[1].set(title="Covariant accel ‖a‖ (0=geodesic)", xlabel="time (s)")
ax[2].plot(tc, kin["curvature"], color="C2"); ax[2].set(title="Curvature κ", xlabel="time (s)")
fig.tight_layout(); display(fig); plt.close(fig)
print(f"geodesic efficiency (endpoint/path) = {kin['efficiency']:.3f}   "
      f"(low ⇒ it wanders/returns rather than going straight)")
print(f"identity speed·dt == step_dist, max error = {ident:.1e}")

Away from the window-edge transients the speed and curvature are roughly flat and
the covariant acceleration is nonzero — **uniform circular motion** on the
manifold, exactly the kinematic signature of a steady rotation. The low geodesic
**efficiency** (endpoint distance ≪ path length) confirms the path loops back
rather than travelling to a new subspace.

## 7. Tangent-PCA — intrinsic dimensionality of the subspace motion

**Tangent-PCA** ("Grassmannian PCA") maps every frame into the flat tangent space
at the trajectory's Fréchet (Karcher) mean via the log map, then runs ordinary
PCA there. The number of components needed to explain ~90% of the variance is an
estimate of the **intrinsic dimensionality** of the subspace's motion. A planar
loop should need **2**.

In [ ]:
from neuralgeom.subspace import tangent_pca, chordal_geodesic, transported_velocities
tp = tangent_pca(emb1["frames"])
arrows = transported_velocities(emb1["frames"], tp)
geo, cho = chordal_geodesic(emb1["frames"], max_pairs=1500)
dim90 = int(np.searchsorted(tp["cum_evr"], 0.90) + 1)

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
ax[0].bar(range(1, len(tp["evr"])+1), tp["evr"], color="C0")
ax[0].plot(range(1, len(tp["evr"])+1), tp["cum_evr"][:len(tp["evr"])], "-o", color="C3")
ax[0].set(title=f"Tangent-PCA spectrum  (dim@90% = {dim90})", xlabel="tangent PC", ylabel="var. frac.")
c = ax[1].scatter(tp["coords"][:,0], tp["coords"][:,1], c=tc, cmap="viridis", s=14)
ax[1].plot(tp["coords"][:,0], tp["coords"][:,1], color="0.85", lw=0.6)
step = max(1, len(arrows)//30)
ax[1].quiver(tp["coords"][:-1:step,0], tp["coords"][:-1:step,1], arrows[::step,0], arrows[::step,1],
             color="C3", width=0.004, alpha=0.7)
ax[1].set(title="Trajectory in tangent PC1–PC2 + transported velocity", xlabel="tPC1", ylabel="tPC2")
fig.colorbar(c, ax=ax[1], label="time (s)")
ax[2].scatter(geo, cho, s=6, alpha=0.3, color="C4")
mx = geo.max() if len(geo) else 1
ax[2].plot([0,mx],[0,mx], "k--", lw=0.8)
ax[2].set(title="Chordal vs geodesic distance", xlabel="geodesic (arc-length)", ylabel="chordal")
fig.tight_layout(); display(fig); plt.close(fig)

The tangent-PCA spectrum is dominated by **two** components and the trajectory in
those two tangent directions is a **closed loop** with a smoothly-rotating velocity
field — the flattened image of the rotation. Chordal vs geodesic distances agree
at small angles and only bend apart for large separations, as expected.

## 8. Topology — persistent homology (the loop, made rigorous)

**Persistent homology (PH)** scans a growing neighbourhood scale over the point
cloud (here, over the *geodesic-distance matrix* of the frames) and records when
topological features are **born** and **die**. A **loop** is an *H1* feature; its
**persistence** (death − birth) is how robust it is. We use 𝔽₂ coefficients — the
field that reveals real-projective / non-orientable structure (`Gr(1,N)=ℝPᴺ⁻¹`).

Two scopes:
- **single** — one trial's trajectory (a within-trial loop);
- **pooled** — frames from all trials together (structure that lives across trials).

In [ ]:
from neuralgeom.subspace import PoolConfig
from neuralgeom.topology.persistence import (single_trial_distances, pooled_distances,
                                             ph, top_life)
from persim import plot_diagrams

Ds = single_trial_distances(traj, tr, EmbedConfig(k=1, win=60, stride=10))
Dp = pooled_distances(traj, PoolConfig(k=1, n_pool=220, fields=False))
dg_s, dg_p = ph(Ds, maxdim=2), ph(Dp, maxdim=2)

fig, ax = plt.subplots(1, 2, figsize=(11, 4.6))
plot_diagrams(dg_s, ax=ax[0]); ax[0].set_title(f"PH — single trial (H1 top {top_life(dg_s[1]):.2f})")
plot_diagrams(dg_p, ax=ax[1]); ax[1].set_title(f"PH — pooled cloud (H1 top {top_life(dg_p[1]):.2f})")
fig.tight_layout(); display(fig); plt.close(fig)

One **H1 point sits far above the diagonal** — a genuinely persistent loop —
in both the single-trial and pooled diagrams. That is the ring, certified without
any embedding (PH runs on distances directly).

### `k` as a topological filter

The same computation at `k = 2` should **not** show the loop: the 2-plane that
contains the rotating line is (approximately) static, so there is nothing circular
to detect. Comparing top-H1 persistence across `k` turns `k` into a *readout of
which part of the computation is moving*.

In [ ]:
rows = []
for k in (1, 2):
    Dsk = single_trial_distances(traj, tr, EmbedConfig(k=k, win=60, stride=10))
    Dpk = pooled_distances(traj, PoolConfig(k=k, n_pool=220, fields=False))
    rows.append((k, top_life(ph(Dsk, maxdim=1)[1]), top_life(ph(Dpk, maxdim=1)[1])))
ks = [r[0] for r in rows]; sing = [r[1] for r in rows]; pool = [r[2] for r in rows]
fig, ax = plt.subplots(figsize=(6, 4))
w = 0.35
ax.bar(np.arange(2)-w/2, sing, w, label="single")
ax.bar(np.arange(2)+w/2, pool, w, label="pooled")
ax.set_xticks([0,1]); ax.set_xticklabels([f"k={k}" for k in ks])
ax.set(title="Top H1 persistence vs k  (the loop is a k=1 phenomenon)", ylabel="H1 persistence")
ax.legend(); fig.tight_layout(); display(fig); plt.close(fig)
print("k, H1_single, H1_pooled:", rows)

## 9. Discrete Exterior Calculus (optional lens)

DEC builds a 2-D simplicial mesh approximating the pooled subspace manifold and
lets us put **scalar fields** on it (energy, subspace speed, input drive,
self-capture) and take their intrinsic gradient/Laplacian. We also show the
**cross-projection matrix** `R[t,j] = ‖Uⱼᵀx(t)‖²/‖x(t)‖²` — how much of the state at
time `t` each visited frame `j` captures; its diagonal is *self-capture* and its
off-diagonal stripes image the loop recurrence directly.

> Caveat (kept honest): the MDS embedding used for the mesh is non-isometric, so
> read the Laplacian *qualitatively*; the persistent homology above is the
> load-bearing topology.

In [ ]:
from neuralgeom.subspace.pooling import pool_frames, PoolConfig
from neuralgeom.topology import dec

frames, fields, _ = pool_frames(traj, PoolConfig(k=1, n_pool=220, fields=True))
XY, tris, tris_pruned, Dpool, stress = dec.build_complex(frames, dec.DECConfig(n_pool=220))
b0, b1, b2 = dec.betti_from_triangles(tris)

fig, ax = plt.subplots(1, 3, figsize=(14, 4.2))
for a, key in zip(ax, ("energy", "speed", "selfcapture")):
    a.triplot(XY[:,0], XY[:,1], tris, color="0.9", lw=0.3)
    sc = a.scatter(XY[:,0], XY[:,1], c=fields[key], s=12, cmap="viridis")
    a.set(title=f"pooled manifold — {key}", xticks=[], yticks=[]); fig.colorbar(sc, ax=a, fraction=0.046)
fig.suptitle(f"DEC mesh   Betti(full) = ({b0},{b1},{b2})   MDS stress={stress:.2g}")
fig.tight_layout(); display(fig); plt.close(fig)

cp = dec.cross_projection(traj, tr, k=1)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
im = ax[0].imshow(cp["R"], origin="lower", aspect="auto", cmap="magma", vmin=0, vmax=1,
                  extent=[cp["win_times"][0], cp["win_times"][-1]]*2)
ax[0].set(title="cross-projection R[t,j]", xlabel="frame time j (s)", ylabel="state time t (s)")
fig.colorbar(im, ax=ax[0], label="captured energy fraction")
ax[1].plot(cp["win_times"], cp["diagonal"], label="self-capture (diag)")
ax[1].plot(cp["win_times"], cp["centrality"], label="frame centrality (mean_t)")
ax[1].set(title="self-capture & centrality", xlabel="time (s)", ylim=(0, 1.02)); ax[1].legend()
fig.tight_layout(); display(fig); plt.close(fig)

## 10. Cross-check against the raw state manifold

Finally, a guard against metric artifacts: estimate the state manifold *directly*
(PCA/Isomap of the raw states) and run the **same** persistent homology on the
Euclidean state distances, then compare its H1 loop to the Grassmannian k=1 loop
(scales normalised before the bottleneck comparison). Agreement between two very
different metrics is strong evidence the loop is real.

In [ ]:
from neuralgeom.topology import direct
r = direct.compare_direct_vs_grassmann(traj, tr, sub=6, maxdim=1)
p3, iso, _ = direct.state_embeddings(traj.X[tr], sub=6)
fig, ax = plt.subplots(1, 3, figsize=(14, 4))
sc = ax[0].scatter(iso[:,0], iso[:,1], c=np.arange(len(iso)), cmap="viridis", s=12)
ax[0].plot(iso[:,0], iso[:,1], color="0.85", lw=0.5)
ax[0].set(title="state-space Isomap(2)", xlabel="dim1", ylabel="dim2")
plot_diagrams(r["dgms_direct"], ax=ax[1]); ax[1].set_title(f"direct state PH (H1 {r['h1_direct']:.2f})")
plot_diagrams(r["dgms_grass"], ax=ax[2]); ax[2].set_title(f"Grassmann k=1 PH (H1 {r['h1_grass']:.2f})")
fig.suptitle(f"scale-normalised H1 bottleneck between the two diagrams = {r['bottleneck']:.3f} "
             f"(smaller ⇒ more similar)")
fig.tight_layout(); display(fig); plt.close(fig)

## 11. What the lens described — summary

For this ring-attractor RNN the subspace lens gave a coherent, cross-validated
account:

- **The coding direction rotates once** — a persistent **H1 loop** on `Gr(1,N)=ℝPᴺ⁻¹`,
  visible in the recurrence matrix, the tangent-PCA loop, and (rigorously) in
  persistent homology, in both single-trial and pooled scopes.
- **`k` is a topological filter**: the loop lives at **k=1**, not k=2 — the *line
  direction* turns while its containing plane is static.
- **Kinematics = uniform circular motion**: near-constant speed and curvature,
  nonzero covariant acceleration, low geodesic efficiency.
- **Intrinsic dimensionality 2** (tangent-PCA), and **high frame reliability** at
  k=1 (large `sv_gap`).
- The **direct state-manifold** cross-check finds the same loop, so it is not an
  artifact of the subspace map.

**Honest limits.** This network was *built* to contain a ring, so recovering it
proves the pipeline is **correct**, not that it is **discriminating** on data with
unknown structure. The topology here has no null-model p-value yet, and the
Grassmannian lens is deliberately blind to computation *within* a fixed subspace
(gain, integration along a fixed axis) — that is the job of the SPD companion and
of null-model inference (see `docs/PROJECTIVE_ROADMAP.md`). The task-trained-RNN
notebook applies the *pullback / dynamics* half of the library to a network whose
structure is **not** hand-built.